# Project #1
## Data Visualization (LECD) 2023/24

#### Student name: Tomás Clemente 2021232197

## Instructions
- Do not modify the structure of this notebook!
- Fill-in the corresponding cells with the code or text depending on the objective
- You can use this template as a working notebook, or copy&paste your final code here
- It is recommended that you export and submit the rendered version of the notebook
- Submit the notebook not later than the date specified on the Inforestudante webpage of the course


## Data
#### Provide a brief description of the chosen dataset and the attributes. If any cleaning or filtering was done, also describe it in this section

No âmbito deste projeto, o dataset escolhido foi sobre bandas metal até 2017. Este dataset contem informação sobre 5000 bandas metal, sendo os atributos de cada uma destas: O nome da banda, o número de fãs da banda, o ano que se formou, o país ou países de origem, o ano que se separaram (caso tenha acontecido) e o estilo ou estilos de metal associados à respetiva banda.

Anteriormente à construção das visualizações, foi necessário ser feito um pré-processamento dos dados e uma filtragem sobre os mesmos. O pré-processamento aplicado concentrou-se na remoção da coluna 'Unnamed: 0' que continha os índices dos dados, a remoção de dados duplicados e na adição de missing data no país ou países de origem de certas bandas, com base em pesquisas. A filtragem foi realizada de forma a usarmos somente os atributos escolhidos para as visualizações sendo estes a origem das bandas, o seu número de fãs e estilos associados, na perspetiva de ser possível visualizar se o país de origem influencia o sucesso da banda tendo em conta o número de estilos que cada banda pratica.

Para facilitar a construção das visualizações dos atributos escolhidos foi realizada uma transformação ao dataframe de modo a que este seja dividido por país (de modo a que as bandas com vários países de formação sejam tomadas em conta) e ainda passar os estilos de variáveis categóricas para numéricas (número de estilos). Para auxiliar nesta operação foi criado manualmente uma lista com todos os países existentes no dataframe.

De modo a facilitar futuras operações de modificação de dataframe criou-se outro.

Um aspeto alterado foi que apenas são consideradas os países com número de fãs superior a 5000 porque, como se pode observar no plot da célula a baixo, existe uma grande dispersão nos dados, o que tornaria as visualizações não percetíveis para os estilos com menos fãs por país. Considera-se assim apenas os países com mais fãs não desfocando do objetivo do trabalho.

Nestes dados é ainda observável que apenas uma pequena percentagem destes é que tem mais do que 4 estilos diferentes, por isso para ser mais fácil de observar substituimos todas as bandas que tinham mais que 4 estilos para 4. Usando depois uma legenda, criada numa nova coluna para que se soubesse que as bandas que têm 4 estilos é 4 ou mais estilos. 

In [1]:
# import your dataset in this section (cell)

#--------------- Importações ---------------#

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


#------------ Pré-Processamento ------------#

df = pd.read_csv('metal_bands_2017_v2.csv') 

df = df.iloc[50:]
df = df.reset_index()
df = df.drop(['index', 'Unnamed: 0'], axis=1)

df['origin'] = df['origin'].astype(str)
df['fans'] = df['fans'].astype(int)

origin_NA = ['Sweden', 'United Kingdom, Romania', 'Israel', 'Sweden, Norway, Ireland, United Kingdom','Austria, United Kingdom, Germany','United Kingdom','Canada, Germany, Italy, USA', 'United Kingdom, Belgium']

a = np.where(df['origin'] == 'nan')

for i in range(len(a[0])):
    df['origin'][a[0][i]] = origin_NA[i]
    

a = np.where(df['origin'] == 'Korea, South')
df['origin'][a[0][0]] = 'South Korea'


#---------- Filtragem de Variáveis ---------#

location = ['Albania', 'Andorra', 'Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahrain', 'Belarus',
             'Belgium', 'Bolivia', 'Bosnia and Herzegovina', 'Brazil', 'Bulgaria', 'Canada', 'Chile', 'China', 'Colombia',
             'Costa Rica', 'Croatia', 'Cyprus', 'Czech Republic', 'Denmark', 'Ecuador', 'Egypt', 'Estonia', 'Faroe Islands',
             'Finland', 'France', 'Georgia', 'Germany', 'Greece', 'Greenland', 'Hungary', 'Iceland', 'India', 'Indonesia', 
             'Iran', 'Iraq', 'Ireland', 'Israel', 'Italy', 'Japan', 'Jersey', 'Jordan', 'Kyrgyzstan', 'Latvia', 'Lebanon', 
             'Liechtenstein', 'Lithuania', 'Luxembourg', 'Macedonia', 'Malaysia', 'Malta', 'Mexico', 'Moldova', 'Monaco', 
             'New Zealand', 'Norway', 'Peru', 'Poland', 'Portugal', 'Romania', 'Russia', 'Saudi Arabia', 'Serbia', 'Singapore', 
             'Slovak Republic', 'Slovenia', 'South Korea', 'South Africa', 'Spain', 'Sri Lanka', 'Swaziland', 'Sweden', 
             'Switzerland', 'Syria', 'Taiwan', 'The Netherlands', 'Tunisia', 'Turkey', 'UAE', 'USA', 'Ukraine', 
             'United Kingdom', 'Venezuela']

df = df.drop(['band_name','formed', 'split'], axis = 1)

for c in location:
    for i in df[df['origin']==c].index:
        #df.loc[i] = [df['fans'][i], c, len(df['style'][i].split(','))]
        df.loc[i] = [df['fans'][i], c, len(df['style'][i].split(','))]
        

for index, entry in df[df['origin'].str.find(',') != -1].iterrows():
    origins = entry['origin'].split(', ')
    #print(origins)
    for origin in origins:
        df.loc[len(df)] = [entry['fans'], origin, len(entry['style'].split(','))]

df = df.drop(df[df['origin'].str.find(',') != -1].index, axis=0)

df['style'] = df['style'].replace([4, 5, 6], 4)


#-------- Dataframe mais específico --------#

df2 = pd.DataFrame(columns = ['fans', 'origin','style'])

for c in location:
    if sum(df[df['origin']==c]['fans'])>5000:
        for s in range(len(df['style'].unique())):
            df2.loc[len(df2)] = [sum(df[df['style']== s+1][df['origin']==c]['fans']), c, s+1]


#----------- Último Processamento ----------#

df2 = df2.drop(np.where(df2['fans'] == 0)[0])

df2 = df2.sort_values(by=['fans'], ascending=False)

df2 = df2.reset_index()

def map_style(value):
    if value in [1, 2, 3]:
        return str(value)
    else:
        return '4+'

df2['label'] = df2['style'].apply(map_style)

df2 = df2.drop(['index','style'], axis=1)


###############################################################################
#------ Plot que não é completamente interpretável (com todos os dados) ------#
###############################################################################

df3 = pd.DataFrame(columns = ['fans', 'origin','style'])
for c in location:
    for s in df[df['origin']==c]['style'].unique():
        df3.loc[len(df3)] = [sum(df[df['origin']==c][df['style'] == s]['fans']),c,s]
        
data = df3.sort_values(by=['fans'], ascending=False)

fig = go.Figure()


for style in data['style'].unique():
    filtered_data = data[data['style'] == style]
    fig.add_trace(go.Scatter(x=filtered_data['origin'], y=filtered_data['fans'],mode='markers', name=str(style)))

fig.update_layout(title="Fans by Band's Style Number (Scatter Plot)",
                title_x=0.5)

fig.update_yaxes(title_text = "Amount of Fans",
    title_font=dict(family='Helvetica', size=13))

fig.update_layout(legend=dict(
    orientation='h',
    yanchor="bottom",
    y=-0.4,
    xanchor="center",
    x=0.5
)) 

fig.show()

C:\Users\tomas\AppData\Local\Temp\ipykernel_2140\137914798.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['origin'][a[0][i]] = origin_NA[i]
C:\Users\tomas\AppData\Local\Temp\ipykernel_2140\137914798.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['origin'][a[0][0]] = 'South Korea'
C:\Users\tomas\AppData\Local\Temp\ipykernel_2140\137914798.py:73: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df2.loc[len(df2)] = [sum(df[df['style']== s+1][df['origin']==c]['fans']), c, s+1]
C:\Users\tomas\AppData\Local\Temp\ipykernel_2140\137914798.py:73: UserWarning: Boolean Series key wil

C:\Users\tomas\AppData\Local\Temp\ipykernel_2140\137914798.py:102: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df3.loc[len(df3)] = [sum(df[df['origin']==c][df['style'] == s]['fans']),c,s]
C:\Users\tomas\AppData\Local\Temp\ipykernel_2140\137914798.py:102: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df3.loc[len(df3)] = [sum(df[df['origin']==c][df['style'] == s]['fans']),c,s]
C:\Users\tomas\AppData\Local\Temp\ipykernel_2140\137914798.py:102: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df3.loc[len(df3)] = [sum(df[df['origin']==c][df['style'] == s]['fans']),c,s]
C:\Users\tomas\AppData\Local\Temp\ipykernel_2140\137914798.py:102: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df3.loc[len(df3)] = [sum(df[df['origin']==c][df['style'] == s]['fans']),c,s]
C:\Users\tomas\AppData\Local\Temp\ipykernel_2140\137914798.py:102: UserWarning: Boolean Series key will be reind

C:\Users\tomas\AppData\Local\Temp\ipykernel_2140\137914798.py:102: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df3.loc[len(df3)] = [sum(df[df['origin']==c][df['style'] == s]['fans']),c,s]
C:\Users\tomas\AppData\Local\Temp\ipykernel_2140\137914798.py:102: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df3.loc[len(df3)] = [sum(df[df['origin']==c][df['style'] == s]['fans']),c,s]
C:\Users\tomas\AppData\Local\Temp\ipykernel_2140\137914798.py:102: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df3.loc[len(df3)] = [sum(df[df['origin']==c][df['style'] == s]['fans']),c,s]
C:\Users\tomas\AppData\Local\Temp\ipykernel_2140\137914798.py:102: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df3.loc[len(df3)] = [sum(df[df['origin']==c][df['style'] == s]['fans']),c,s]
C:\Users\tomas\AppData\Local\Temp\ipykernel_2140\137914798.py:102: UserWarning: Boolean Series key will be reind

## Visualization model #1

In [2]:
# implementation of the viz 1 in this cell

#Scatter
data = df2.sort_values('fans', ascending=False)

fig = go.Figure()


for style in data['label'].unique():
    filtered_data = data[data['label'] == style]
    fig.add_trace(go.Scatter(x=filtered_data['origin'], y=filtered_data['fans'],mode='markers', name=style))

fig.update_layout(title="Fans by Band's Style Number (Scatter Plot)",
                title_x=0.5)

fig.update_yaxes(title_text = "Amount of Fans",
    title_font=dict(family='Helvetica', size=13))

fig.update_layout(legend=dict(
    orientation='h',
    yanchor="bottom",
    y=-0.4,
    xanchor="center",
    x=0.5
)) 

fig.show()

## Visualization model #2

In [3]:
# implementation of the viz 2 in this cell

#Stacked Bar Chart
fig = go.Figure()

data = df2.sort_values('fans', ascending=False)

for style in data['label'].unique():
    filtered_data = data[data['label'] == style]
    fig.add_trace(go.Bar(x=filtered_data['origin'], y=filtered_data['fans'], name=style))

fig.update_layout(title="Fans by Band's Style Number (Stacked Bar Chart)",
                barmode='relative', title_x=0.5)

fig.update_yaxes(title_text = "Amount of Fans",
    title_font=dict(family='Helvetica', size=13))

fig.update_layout(legend=dict(
    orientation='h',
    yanchor="bottom",
    y=-0.4,
    xanchor="center",
    x=0.5
))    
    
    
fig.update_layout(hovermode="x unified")   
 

fig.show()

## Visualization model #3

In [4]:
# implementation of the viz 3 in this cell

#Pie Chart

specs =[[{'type':'pie'}]*4]*3   # *cols *rows
subplot_titles = []
for c in data['origin'].unique():
    subplot_titles.append(c)

fig = make_subplots(rows=3, cols=4, specs =specs, subplot_titles=subplot_titles)

i=0
for c in data['origin'].unique():
    row = i // 4 + 1
    col = i % 4 + 1
    fig.add_trace(go.Pie(labels=data[data['origin']==c]['label'], values=data[data['origin']==c]['fans'], name =c),row, col)
    
    i+=1
    
fig.update_layout(title="Fans by Band's Style Number (Pie Chart)",
                title_x=0.5)

fig.show()

## Visualization model #4

In [5]:
# implementation of the viz 4 in this cell

# Parallel Coordinates Plot

text_ori = [ori for ori in df2['origin'].unique()]
vals_ori = [o for o in range(len(df2['origin'].unique()))] 
text_lab = [lab for lab in df2['label'].unique()]
vals_lab = [l for l in range(len(df2['label'].unique()))] 

fig = go.Figure(data=
    go.Parcoords(line_color='blue',
        dimensions = list([
            dict(range = [0, len(df2['origin'].unique())-1],
                 tickvals = vals_ori,
                 label = 'Origin', values = pd.factorize(data['origin'])[0],
                 ticktext = list(pd.factorize(data['origin'])[1])
                ),
            dict(range = [1, max(data['fans'])], label = 'Fãs', values = data['fans']),
            dict(range = [0, len(df2['label'].unique())],
                 tickvals = vals_lab,
                 label = 'Number_Styles', values = pd.factorize(data['label'])[0],
                 ticktext = list(pd.factorize(data['label'])[1])
                )
                
        ])
    )
)

fig.update_layout(title="Fans by Band's Style Number (Parallel Coordinates Plot)",
                title_x=0.5)
fig.show()

## Critical reflection
#### Write down 2-3 paragraphs (190-240 words) with your reflection about the obtained visualizations.

Considerando as representações visuais elaboradas, o Stacked Bar Chart destaca-se como a melhor opção para comunicar o objetivo inicial do estudo, devido à sua capacidade de evidenciar as diferenças de fãs entre estilos musicais e países de origem das bandas de forma clara. O Pie Chart, embora memorável, não oferece a mesma profundidade de informação sobre as variações por país e quantidades de estilo, pois esta representação mostra os dados por cada país (não tendo a comparação entre países). Para explorar os dados com mais detalhes, o Parallel Coordinates Plot é o mais fiável, permitindo a seleção de dimensões para análise, como o número de estilos, países e número de fãs a ser estudado. Permite ainda identificar facilmente tendências relevantes.

Em relação ao Scatter Plot, embora seja útil para observar as diferenças de fãs entre estilos e países, não é tão transparente de analisar como o Stacked Bar Chart. Ambos acabam por ser muito parecidos, com esta pequena diferença ao serem analisados. 

Após as visualizações dos dados, estes sugerem que bandas com menos estilos musicais tendem a ter mais fãs, e a origem do país, particularmente em países de língua inglesa, também influencia o número de fãs para uma quantidade maior. Portanto, o Stacked Bar Chart é a escolha preferencial para comunicar com o cliente, enquanto o Parallel Coordinates Plot é o mais indicado para uma exploração mais aprofundada dos dados.


## References
### All the sources you have used in this project should cited and credited in this section.


###### Sites de Pesquisa para Tratamento dos dados:
- https://www.tutorialspoint.com/index.htm
- https://pandas.pydata.org/docs/index.html
- https://www.geeksforgeeks.org/
- https://plotly.com/python/


###### Sites de auxilio com as Visualizações:
- https://datavizproject.com/
- https://plotly.com/python/line-and-scatter/
- https://plotly.com/python/bar-charts/
- https://plotly.com/python/pie-charts/
- https://plotly.com/python/parallel-categories-diagram/
- https://community.plotly.com/